# Hey BMO wake-word training
Pinned openWakeWord v0.5.1 pipeline. Run top to bottom on a GPU runtime.
Edit only `config.yaml` to change the phrase. See README.md for the model contract.

## 1. Setup: install pinned deps + tools

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# piper-sample-generator has no PyPI release; pin to a known-good commit.
PIPER_COMMIT = 'b8d72e5'
if not os.path.isdir('piper-sample-generator'):
    subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper-sample-generator'], check=True)
    subprocess.run(['git', '-C', 'piper-sample-generator', 'checkout', PIPER_COMMIT], check=True)
print('deps installed')

In [ ]:
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
print('phrase:', cfg['target_phrase'])
print('model :', cfg['model_name'])
print('samples:', cfg['n_samples'], 'val:', cfg['n_samples_val'])

## 2. Download openWakeWord training data (negatives, RIRs, noise)

In [ ]:
# openWakeWord ships helpers to fetch its precomputed negative features and
# augmentation sets. This downloads several GB on first run (cached after).
import openwakeword
from openwakeword import train
train.download_training_data() if hasattr(train, 'download_training_data') else print(
    'Use the download cells from the pinned openWakeWord automatic_model_training notebook')
print('openWakeWord', openwakeword.__version__)

## 3. Audition: synthesize a small batch and listen
If the pronunciation is wrong, edit `target_phrase` in `config.yaml`, re-run cell 1's config load, then re-run this cell.

In [ ]:
import subprocess, glob
os.makedirs('audition', exist_ok=True)
subprocess.run([sys.executable, 'piper-sample-generator/generate_samples.py',
                cfg['target_phrase'][0], '--max-samples', '10',
                '--output-dir', 'audition'], check=True)
from IPython.display import Audio, display
for w in sorted(glob.glob('audition/*.wav'))[:5]:
    print(w); display(Audio(w))

## 4. Synthesize → augment → train (reads config.yaml)

In [ ]:
# openWakeWord's automatic training entrypoint. It synthesizes positives via
# piper, augments with the RIR/noise sets, computes base mel->embedding
# features, trains the classifier, and exports ONNX into cfg['output_dir'].
subprocess.run([sys.executable, '-m', 'openwakeword.train',
                '--training_config', 'config.yaml',
                '--generate_clips', '--augment_clips', '--train_model'], check=True)

## 5. Locate the exported model + print metrics

In [ ]:
import glob
onnx = sorted(glob.glob(os.path.join(cfg['output_dir'], '**', cfg['model_name'] + '.onnx'), recursive=True))
assert onnx, 'no exported ONNX found; check the training cell output'
print('exported:', onnx[-1])
print('Download this file and validate with cmd/wakeword-eval (see README.md),')
print('then copy it to assets/wakeword/hey_bmo.onnx in the repo.')